# Extracción de Detecciones por Nodo desde DetalleTodasCorridas.xlsx

**Objetivo:** Convertir las hojas del archivo Excel `DetalleTodasCorridas.xlsx` a archivos CSV individuales para compatibilidad con el pipeline de remapeo.

**Contexto:**
- Simulaciones 2026 generaron `DetalleTodasCorridas.xlsx` con 1,080 hojas (abolladuras) o 2,160 hojas (corrosión)
- Cada hoja `ID_XXXX` contiene: `Numero_de_nodo`, `Valor_de_daño_normalizado`, `Estado`
- Pipeline de remapeo requiere archivos CSV individuales en carpeta `salida_csvs/`

**Estructura de salida:**
```
outputs/resultados_nuevos/
├── abolladuras_2026/
│   └── salida_csvs/
│       ├── ID_0001.csv
│       ├── ID_0002.csv
│       └── ...
└── corrosion_2026/
    └── salida_csvs/
        ├── ID_0001.csv
        ├── ID_0002.csv
        └── ...
```

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas exitosamente")

## 1. Configuración de Rutas

In [ ]:
# Rutas absolutas al workspace
base_path = Path.home() / 'github' / 'Proyecto-doctoral'

# Archivos Excel de entrada
excel_abolladura = base_path / 'Resultados' / 'abolladura_2026-02-26_05-26-02' / 'DetalleTodasCorridas.xlsx'
excel_corrosion = base_path / 'Resultados' / 'corrosion_2026-02-27_06-07-17' / 'DetalleTodasCorridas.xlsx'

# Directorios de salida
output_abolladura = base_path / 'outputs' / 'resultados_nuevos' / 'abolladuras_2026' / 'salida_csvs'
output_corrosion = base_path / 'outputs' / 'resultados_nuevos' / 'corrosion_2026' / 'salida_csvs'

# Crear directorios si no existen
output_abolladura.mkdir(parents=True, exist_ok=True)
output_corrosion.mkdir(parents=True, exist_ok=True)

print(f"✓ Excel abolladuras: {excel_abolladura.exists()}")
print(f"✓ Excel corrosión: {excel_corrosion.exists()}")
print(f"✓ Directorio salida abolladuras: {output_abolladura}")
print(f"✓ Directorio salida corrosión: {output_corrosion}")

## 2. Función de Extracción

In [ ]:
def extraer_hojas_a_csv(excel_path, output_dir, n_corridas):
    """
    Extrae hojas de DetalleTodasCorridas.xlsx a archivos CSV individuales.
    
    Parámetros:
    -----------
    excel_path : Path
        Ruta al archivo Excel de entrada
    output_dir : Path
        Directorio donde se guardarán los CSVs
    n_corridas : int
        Número de corridas (hojas) a procesar
    
    Retorna:
    --------
    dict : Estadísticas del procesamiento
    """
    stats = {
        'procesadas': 0,
        'errores': 0,
        'archivos_generados': []
    }
    
    print(f"\n{'='*70}")
    print(f"Procesando: {excel_path.name}")
    print(f"{'='*70}\n")
    
    for i in tqdm(range(1, n_corridas + 1), desc="Extrayendo hojas"):
        sheet_name = f'ID_{i:04d}'
        output_file = output_dir / f'{sheet_name}.csv'
        
        try:
            # Leer hoja específica
            df = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')
            
            # Verificar columnas esperadas
            expected_cols = ['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']
            if not all(col in df.columns for col in expected_cols):
                print(f"\n⚠️  Advertencia: Hoja {sheet_name} tiene columnas inesperadas")
                print(f"    Esperadas: {expected_cols}")
                print(f"    Encontradas: {list(df.columns)}")
            
            # Guardar como CSV
            df.to_csv(output_file, index=False)
            
            stats['procesadas'] += 1
            stats['archivos_generados'].append(output_file.name)
            
        except Exception as e:
            print(f"\n❌ Error procesando {sheet_name}: {str(e)}")
            stats['errores'] += 1
    
    return stats

## 3. Extracción: Abolladuras (1,080 corridas)

In [ ]:
stats_abolladura = extraer_hojas_a_csv(
    excel_path=excel_abolladura,
    output_dir=output_abolladura,
    n_corridas=1080
)

print(f"\n{'='*70}")
print("RESUMEN - ABOLLADURAS")
print(f"{'='*70}")
print(f"✓ Hojas procesadas: {stats_abolladura['procesadas']}")
print(f"✗ Errores: {stats_abolladura['errores']}")
print(f"📁 Archivos en: {output_abolladura}")
print(f"\nPrimeros 5 archivos: {stats_abolladura['archivos_generados'][:5]}")
print(f"Últimos 5 archivos: {stats_abolladura['archivos_generados'][-5:]}")

## 4. Extracción: Corrosión (2,160 corridas)

In [ ]:
stats_corrosion = extraer_hojas_a_csv(
    excel_path=excel_corrosion,
    output_dir=output_corrosion,
    n_corridas=2160
)

print(f"\n{'='*70}")
print("RESUMEN - CORROSIÓN")
print(f"{'='*70}")
print(f"✓ Hojas procesadas: {stats_corrosion['procesadas']}")
print(f"✗ Errores: {stats_corrosion['errores']}")
print(f"📁 Archivos en: {output_corrosion}")
print(f"\nPrimeros 5 archivos: {stats_corrosion['archivos_generados'][:5]}")
print(f"Últimos 5 archivos: {stats_corrosion['archivos_generados'][-5:]}")

## 5. Verificación de Consistencia

In [ ]:
# Verificar que los archivos se generaron correctamente
csv_abolladura = list(output_abolladura.glob('ID_*.csv'))
csv_corrosion = list(output_corrosion.glob('ID_*.csv'))

print(f"\n{'='*70}")
print("VERIFICACIÓN FINAL")
print(f"{'='*70}")
print(f"\n✓ Abolladuras: {len(csv_abolladura)} archivos CSV (esperados: 1,080)")
print(f"✓ Corrosión: {len(csv_corrosion)} archivos CSV (esperados: 2,160)")

# Leer archivo de ejemplo para verificar estructura
if len(csv_abolladura) > 0:
    ejemplo_abol = pd.read_csv(csv_abolladura[0])
    print(f"\n📊 Estructura de ejemplo (abolladura ID_0001):")
    print(f"   Dimensiones: {ejemplo_abol.shape}")
    print(f"   Columnas: {list(ejemplo_abol.columns)}")
    print(f"\n   Primeras 5 filas:")
    print(ejemplo_abol.head())
    
    # Verificar nodos con daño
    nodos_dano = ejemplo_abol[ejemplo_abol['Estado'] == 'Daño']
    print(f"\n   Nodos con Estado='Daño': {len(nodos_dano)}")
    if len(nodos_dano) > 0:
        print(f"   Valores de daño normalizado: {nodos_dano['Valor_de_daño_normalizado'].describe()}")

if len(csv_corrosion) > 0:
    ejemplo_corr = pd.read_csv(csv_corrosion[0])
    print(f"\n📊 Estructura de ejemplo (corrosion ID_0001):")
    print(f"   Dimensiones: {ejemplo_corr.shape}")
    print(f"   Columnas: {list(ejemplo_corr.columns)}")
    
    nodos_dano_corr = ejemplo_corr[ejemplo_corr['Estado'] == 'Daño']
    print(f"\n   Nodos con Estado='Daño': {len(nodos_dano_corr)}")

print(f"\n{'='*70}")
print("✅ Extracción completada exitosamente")
print(f"{'='*70}")

## 📝 Notas

**Archivos generados:**
- `outputs/resultados_nuevos/abolladuras_2026/salida_csvs/` → 1,080 CSVs
- `outputs/resultados_nuevos/corrosion_2026/salida_csvs/` → 2,160 CSVs

**Formato de cada CSV:**
```csv
Numero_de_nodo,Valor_de_daño_normalizado,Estado
5,99.988311,Daño
6,80.878867,Daño
7,5.234567,-
...
```

**Próximos pasos:**
1. Copiar archivos auxiliares (`nodos_beams.csv`, `nodos_inclined_legs.csv`)
2. Aplicar remapeo 0.5 con notebooks de remapeo
3. Calcular ICD con datos remapeados
4. Análisis estadístico de vectores alpha